In [10]:
# Import necessary libraries
import os
import kagglehub
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [12]:
# Download the datasets
path1 = kagglehub.dataset_download("mamun1113/doctors-handwritten-prescription-bd-dataset")
print("Path to Dataset 1 files:", path1)
path2 = kagglehub.dataset_download("mehaksingal/illegible-medical-prescription-images-dataset")
print("Path to Dataset 2 files:", path2)

100%|██████████████████████████████████████████████████████████████████████████████| 19.1M/19.1M [00:06<00:00, 3.31MB/s]

Extracting files...


Path to Dataset 1 files: /Users/dinethpanditha/.cache/kagglehub/datasets/mamun1113/doctors-handwritten-prescription-bd-dataset/versions/1


100%|██████████████████████████████████████████████████████████████████████████████| 24.5M/24.5M [00:07<00:00, 3.27MB/s]

Extracting files...
Path to Dataset 2 files: /Users/dinethpanditha/.cache/kagglehub/datasets/mehaksingal/illegible-medical-prescription-images-dataset/versions/1


In [28]:
# Set random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Define dataset paths (replace with your actual paths)
dataset1_path = "/Users/dinethpanditha/.cache/kagglehub/datasets/mamun1113/doctors-handwritten-prescription-bd-dataset/versions/1"
dataset2_path = "/Users/dinethpanditha/.cache/kagglehub/datasets/mehaksingal/illegible-medical-prescription-images-dataset/versions/1"

In [36]:
# -------------------
# 2.1 Data Import and Initial Assessment
# -------------------
# Recursive image loading function (no print for failed loads)
def load_images_from_folder_recursive(folder):
    images = []
    filenames = []
    for root, _, files in os.walk(folder):
        for filename in files:
            if filename.lower().endswith(('.jpg', '.png', '.jpeg')):
                img_path = os.path.join(root, filename)
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    images.append(img)
                    filenames.append(filename)
                # Silently skip failed loads (no print)
    return images, filenames

# Load datasets
images1, filenames1 = load_images_from_folder_recursive(dataset1_path)
images2, filenames2 = load_images_from_folder_recursive(dataset2_path)

# Combine datasets
all_images = images1 + images2
all_filenames = filenames1 + filenames2

# Initial assessment
print(f"Total images loaded: {len(all_images)}")
print(f"Sample image shape: {all_images[0].shape if all_images else 'No images'}")
blurry_count = sum(1 for img in all_images if cv2.Laplacian(img, cv2.CV_64F).var() < 100)
print(f"Number of blurry images (variance < 100): {blurry_count}")

Total images loaded: 4802
Sample image shape: (69, 110)
Number of blurry images (variance < 100): 9


In [38]:
# -------------------
# 2.2 Data Cleaning and Pre-processing
# -------------------
def preprocess_image(img):
    # Resize to 128x128
    img = cv2.resize(img, (128, 128))
    # Remove noise with Gaussian blur
    img = cv2.GaussianBlur(img, (5, 5), 0)
    # Threshold to binarize (enhance contrast) - keep as 8-bit for OTSU
    _, img = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Normalize pixel values after thresholding (0 to 1 for CNN)
    img = img / 255.0
    return img

# Apply preprocessing and filter out blurry images
cleaned_images = [preprocess_image(img) for img in all_images if cv2.Laplacian(img, cv2.CV_64F).var() >= 100]
print(f"Images after cleaning (removed blurry): {len(cleaned_images)}")

Images after cleaning (removed blurry): 4793


In [42]:
# -------------------
# 2.3 Data Preparation and Feature Engineering
# -------------------
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Ensure cleaned_images and all_filenames are defined from previous sections
# Simulate labels (binary: legible=1, illegible=0) based on filenames if possible
# Check filenames for keywords; fallback to random if needed
def generate_labels(filenames, num_images):
    labels = []
    for fname in filenames[:num_images]:  # Match length of cleaned_images
        fname_lower = fname.lower()
        if "legible" in fname_lower or "clear" in fname_lower:
            labels.append(1)  # Legible
        elif "illegible" in fname_lower or "bad" in fname_lower:
            labels.append(0)  # Illegible
        else:
            labels.append(np.random.randint(0, 2))  # Random if no keyword
    return labels

# Generate labels
labels = generate_labels(all_filenames, len(cleaned_images))

# Verify lengths match
if len(labels) != len(cleaned_images):
    raise ValueError(f"Label count ({len(labels)}) does not match image count ({len(cleaned_images)})")

# Convert to numpy arrays and reshape for CNN (add channel dimension)
X = np.array(cleaned_images, dtype=np.float32).reshape(-1, 128, 128, 1)  # Ensure float32 for CNN
y = np.array(labels, dtype=np.int32)  # Binary labels as integers

# Data augmentation to improve model generalization
datagen = ImageDataGenerator(
    rotation_range=10,         # Small rotations to mimic handwriting tilt
    width_shift_range=0.1,     # Horizontal shifts
    height_shift_range=0.1,    # Vertical shifts
    zoom_range=0.1,           # Slight zoom for robustness
    fill_mode='nearest'        # Fill gaps with nearest pixel
)
datagen.fit(X)

print(f"Prepared data: X shape = {X.shape}, y shape = {y.shape}")

Prepared data: X shape = (4793, 128, 128, 1), y shape = (4793,)


In [46]:
# -------------------
# 2.4 Exploratory Data Analysis
# -------------------
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# Ensure output directory exists
output_dir = "eda_plots"
os.makedirs(output_dir, exist_ok=True)

# Verify inputs
if not cleaned_images or not labels:
    raise ValueError("cleaned_images or labels are empty. Check previous steps.")

# Convert labels to numpy array to avoid FutureWarning
labels_np = np.array(labels)

# Distribution of labels
plt.figure(figsize=(8, 5))
sns.countplot(x=labels_np, palette="viridis")
plt.title("Distribution of Legible vs. Illegible Prescriptions", fontsize=12)
plt.xlabel("Label (0 = Illegible, 1 = Legible)", fontsize=10)
plt.ylabel("Count", fontsize=10)
plt.xticks([0, 1], ["Illegible", "Legible"])
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(os.path.join(output_dir, "label_distribution.png"), dpi=300, bbox_inches='tight')
plt.close()

# Sample image visualization
num_samples = min(4, len(cleaned_images))
plt.figure(figsize=(12, 6))
for i in range(num_samples):
    plt.subplot(2, 2, i + 1)
    plt.imshow(cleaned_images[i], cmap='gray')
    plt.title(f"Sample {i + 1}: Label = {'Legible' if labels_np[i] == 1 else 'Illegible'}", fontsize=10)
    plt.axis('off')
plt.suptitle("Sample Preprocessed Prescription Images", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(output_dir, "sample_images.png"), dpi=300, bbox_inches='tight')
plt.close()

# Blurriness distribution
blurriness = [cv2.Laplacian(img, cv2.CV_64F).var() for img in cleaned_images]
plt.figure(figsize=(8, 5))
plt.hist(blurriness, bins=20, color='skyblue', edgecolor='black')
plt.title("Distribution of Image Blurriness (Laplacian Variance)", fontsize=12)
plt.xlabel("Blurriness (Variance)", fontsize=10)
plt.ylabel("Count", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(os.path.join(output_dir, "blurriness_distribution.png"), dpi=300, bbox_inches='tight')
plt.close()

print(f"EDA plots saved in '{output_dir}' directory.")

EDA plots saved in 'eda_plots' directory.


In [52]:
# -------------------
# 2.5 Model Development and Experiments
# -------------------
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
import numpy as np

# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Verify dataset size
if X.shape[0] < 50:
    raise ValueError(f"Dataset too small ({X.shape[0]} samples). Need more data.")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]} samples, Test set: {X_test.shape[0]} samples")

# Define CNN model with regularization
def build_cnn():
    model = Sequential([
        Input(shape=(128, 128, 1)),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),  # L2 regularization
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),  # Increase LR
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    return model

# Early stopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Experiment 1: Basic CNN
model1 = build_cnn()
history1 = model1.fit(
    X_train, y_train,
    epochs=20,  # More epochs, but early stopping will halt if needed
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
    verbose=1
)

# Experiment 2: CNN with stronger augmentation
datagen = ImageDataGenerator(
    rotation_range=20,  # Increase for more variety
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    shear_range=10,
    fill_mode='nearest'
)
datagen.fit(X_train)

model2 = build_cnn()
history2 = model2.fit(
    datagen.flow(X_train, y_train, batch_size=32, seed=42),
    epochs=20,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
    verbose=1
)

# Save models
model1.save('basic_cnn_model.keras')
model2.save('augmented_cnn_model.keras')
print("Models saved as 'basic_cnn_model.keras' and 'augmented_cnn_model.keras'")

Training set: 3834 samples, Test set: 959 samples
Epoch 1/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 33s 268ms/step - accuracy: 0.4958 - loss: 8.9652 - precision_4: 0.5128 - recall_4: 0.4986 - val_accuracy: 0.5016 - val_loss: 3.8454 - val_precision_4: 0.5010 - val_recall_4: 0.9979
Epoch 2/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 33s 273ms/step - accuracy: 0.5152 - loss: 2.3246 - precision_4: 0.5395 - recall_4: 0.4185 - val_accuracy: 0.5005 - val_loss: 2.5939 - val_precision_4: 0.5005 - val_recall_4: 1.0000
Epoch 3/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 32s 269ms/step - accuracy: 0.4658 - loss: 1.2623 - precision_4: 0.4720 - recall_4: 0.3589 - val_accuracy: 0.5016 - val_loss: 1.3975 - val_precision_4: 0.5011 - val_recall_4: 0.9917
Epoch 4/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 32s 267ms/step - accuracy: 0.4912 - loss: 1.0565 - precision_4: 0.5055 - recall_4: 0.4577 - val_accuracy: 0.5005 - val_loss: 1.0408 - val_precision_4: 0.5006 - val_recall_4: 0.9229
Epoch 5/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 31s 261ms/step - accuracy:

/opt/anaconda3/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


120/120 ━━━━━━━━━━━━━━━━━━━━ 35s 279ms/step - accuracy: 0.5165 - loss: 9.0060 - precision_5: 0.5193 - recall_5: 0.4942 - val_accuracy: 0.4995 - val_loss: 6.1376 - val_precision_5: 0.0000e+00 - val_recall_5: 0.0000e+00
Epoch 2/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 34s 281ms/step - accuracy: 0.5011 - loss: 2.6186 - precision_5: 0.5032 - recall_5: 0.5993 - val_accuracy: 0.5068 - val_loss: 3.0754 - val_precision_5: 0.5037 - val_recall_5: 0.9958
Epoch 3/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 36s 303ms/step - accuracy: 0.5090 - loss: 1.6389 - precision_5: 0.5205 - recall_5: 0.7233 - val_accuracy: 0.5016 - val_loss: 1.3743 - val_precision_5: 0.5039 - val_recall_5: 0.2667
Epoch 4/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 32s 270ms/step - accuracy: 0.4996 - loss: 1.1405 - precision_5: 0.5138 - recall_5: 0.6913 - val_accuracy: 0.4984 - val_loss: 0.9375 - val_precision_5: 0.4977 - val_recall_5: 0.2250
Epoch 5/20
120/120 ━━━━━━━━━━━━━━━━━━━━ 33s 278ms/step - accuracy: 0.5008 - loss: 0.8718 - precision_5: 0.5015 - recall

In [54]:
# -------------------
# 2.6 Model Evaluation
# -------------------
# Evaluate models
y_pred1 = (model1.predict(X_test) > 0.5).astype(int)
y_pred2 = (model2.predict(X_test) > 0.5).astype(int)

# Accuracy
acc1 = accuracy_score(y_test, y_pred1)
acc2 = accuracy_score(y_test, y_pred2)
print(f"Basic CNN Accuracy: {acc1:.2f}")
print(f"Augmented CNN Accuracy: {acc2:.2f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred2)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix (Augmented CNN)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.savefig("confusion_matrix.png")
plt.close()

# Training history plot
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history2.history['accuracy'], label='Train Acc')
plt.plot(history2.history['val_accuracy'], label='Val Acc')
plt.title("Accuracy Over Epochs")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history2.history['loss'], label='Train Loss')
plt.plot(history2.history['val_loss'], label='Val Loss')
plt.title("Loss Over Epochs")
plt.legend()
plt.savefig("training_history.png")
plt.close()

# Save models (optional)
model2.save("augmented_cnn_model.h5")
print("Model saved as 'augmented_cnn_model.h5'")


30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step
Basic CNN Accuracy: 0.50
Augmented CNN Accuracy: 0.50


Model saved as 'augmented_cnn_model.h5'


In [58]:
# 2.6 Model Evaluation
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import os

output_dir = "evaluation_plots"
os.makedirs(output_dir, exist_ok=True)

y_pred1 = (model1.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
y_pred2 = (model2.predict(X_test, verbose=0) > 0.5).astype(int).flatten()

metrics = {
    'Basic CNN': {
        'Accuracy': accuracy_score(y_test, y_pred1),
        'Precision': precision_score(y_test, y_pred1),
        'Recall': recall_score(y_test, y_pred1),
        'F1-Score': f1_score(y_test, y_pred1)
    },
    'Augmented CNN': {
        'Accuracy': accuracy_score(y_test, y_pred2),
        'Precision': precision_score(y_test, y_pred2),
        'Recall': recall_score(y_test, y_pred2),
        'F1-Score': f1_score(y_test, y_pred2)
    }
}

print("Model Evaluation Metrics:")
for model_name, scores in metrics.items():
    print(f"\n{model_name}:")
    for metric, value in scores.items():
        print(f"{metric}: {value:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
cm1 = confusion_matrix(y_test, y_pred1)
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues', ax=ax1)
ax1.set_title("Confusion Matrix (Basic CNN)")
ax1.set_xlabel("Predicted")
ax1.set_ylabel("Actual")

cm2 = confusion_matrix(y_test, y_pred2)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues', ax=ax2)
ax2.set_title("Confusion Matrix (Augmented CNN)")
ax2.set_xlabel("Predicted")
ax2.set_ylabel("Actual")

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "confusion_matrices.png"), dpi=300)
plt.close()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].plot(history1.history['accuracy'], label='Train Acc')
axes[0, 0].plot(history1.history['val_accuracy'], label='Val Acc')
axes[0, 0].set_title("Basic CNN: Accuracy Over Epochs")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Accuracy")
axes[0, 0].legend()
axes[0, 0].grid(True, linestyle='--', alpha=0.7)

axes[0, 1].plot(history1.history['loss'], label='Train Loss')
axes[0, 1].plot(history1.history['val_loss'], label='Val Loss')
axes[0, 1].set_title("Basic CNN: Loss Over Epochs")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Loss")
axes[0, 1].legend()
axes[0, 1].grid(True, linestyle='--', alpha=0.7)

axes[1, 0].plot(history2.history['accuracy'], label='Train Acc')
axes[1, 0].plot(history2.history['val_accuracy'], label='Val Acc')
axes[1, 0].set_title("Augmented CNN: Accuracy Over Epochs")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Accuracy")
axes[1, 0].legend()
axes[1, 0].grid(True, linestyle='--', alpha=0.7)

axes[1, 1].plot(history2.history['loss'], label='Train Loss')
axes[1, 1].plot(history2.history['val_loss'], label='Val Loss')
axes[1, 1].set_title("Augmented CNN: Loss Over Epochs")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Loss")
axes[1, 1].legend()
axes[1, 1].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "training_history.png"), dpi=300)
plt.close()

print(f"Evaluation plots saved in '{output_dir}' directory.")

Model Evaluation Metrics:

Basic CNN:
Accuracy: 0.5016
Precision: 0.5010
Recall: 1.0000
F1-Score: 0.6676

Augmented CNN:
Accuracy: 0.5036
Precision: 0.8333
Recall: 0.0104
F1-Score: 0.0206
Evaluation plots saved in 'evaluation_plots' directory.


## =======

In [61]:
# 2.6 Model Evaluation
output_dir = "evaluation_plots"
os.makedirs(output_dir, exist_ok=True)

y_pred1 = (model1.predict(X_test, verbose=0) > 0.5).astype(int).flatten()
y_pred2 = (model2.predict(X_test, verbose=0) > 0.5).astype(int).flatten()

metrics = {
    'Basic CNN': {
        'Accuracy': accuracy_score(y_test, y_pred1),
        'Precision': precision_score(y_test, y_pred1),
        'Recall': recall_score(y_test, y_pred1),
        'F1-Score': f1_score(y_test, y_pred1)
    },
    'Augmented CNN': {
        'Accuracy': accuracy_score(y_test, y_pred2),
        'Precision': precision_score(y_test, y_pred2),
        'Recall': recall_score(y_test, y_pred2),
        'F1-Score': f1_score(y_test, y_pred2)
    }
}

print("Model Evaluation Metrics:")
for model_name, scores in metrics.items():
    print(f"\n{model_name}:")
    for metric, value in scores.items():
        print(f"{metric}: {value:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
cm1 = confusion_matrix(y_test, y_pred1)
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues', ax=ax1)
ax1.set_title("Confusion Matrix (Basic CNN)")
ax1.set_xlabel("Predicted")
ax1.set_ylabel("Actual")

cm2 = confusion_matrix(y_test, y_pred2)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues', ax=ax2)
ax2.set_title("Confusion Matrix (Augmented CNN)")
ax2.set_xlabel("Predicted")
ax2.set_ylabel("Actual")

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "confusion_matrices.png"), dpi=300)
plt.close()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].plot(history1.history['accuracy'], label='Train Acc')
axes[0, 0].plot(history1.history['val_accuracy'], label='Val Acc')
axes[0, 0].set_title("Basic CNN: Accuracy Over Epochs")
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(history1.history['loss'], label='Train Loss')
axes[0, 1].plot(history1.history['val_loss'], label='Val Loss')
axes[0, 1].set_title("Basic CNN: Loss Over Epochs")
axes[0, 1].legend()
axes[0, 1].grid(True)

axes[1, 0].plot(history2.history['accuracy'], label='Train Acc')
axes[1, 0].plot(history2.history['val_accuracy'], label='Val Acc')
axes[1, 0].set_title("Augmented CNN: Accuracy Over Epochs")
axes[1, 0].legend()
axes[1, 0].grid(True)

axes[1, 1].plot(history2.history['loss'], label='Train Loss')
axes[1, 1].plot(history2.history['val_loss'], label='Val Loss')
axes[1, 1].set_title("Augmented CNN: Loss Over Epochs")
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "training_history.png"), dpi=300)
plt.close()

print(f"Evaluation plots saved in '{output_dir}' directory.")

Model Evaluation Metrics:

Basic CNN:
Accuracy: 0.5016
Precision: 0.5010
Recall: 1.0000
F1-Score: 0.6676

Augmented CNN:
Accuracy: 0.5036
Precision: 0.8333
Recall: 0.0104
F1-Score: 0.0206
Evaluation plots saved in 'evaluation_plots' directory.
